# 01 STRING data, identifiers and frozen analysis choices

Let $L\in\{F,P\}$ denote the functional and physical STRING layers. For proteins $i,j$, let $c_{ij}^{(L)}$ be the downloaded integer `combined_score`.

At threshold $\tau$, define

$$
A_{ij}^{(L,\tau)}=1_{\{c_{ij}^{(L)}\ge1000\tau\}},\quad W_{ij}^{(L,\tau)}=\frac{c_{ij}^{(L)}}{1000}A_{ij}^{(L,\tau)}
$$

$A$ records whether an edge is retained; $W$ records its confidence score. The score is confidence in an association, not the strength or direction of a biological effect.

1. Verify the 4 raw files and record their hashes;
2. inspect their columns;
3. create one undirected edge table per network layer;
4. check identifier coverage, especially HLJ1;
5. create a manually curated ERAD seed-table template;
6. summarise network coverage at $\tau=0.4,0.7,0.9$

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import re

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw" / "string_v12_0"
PROCESSED = ROOT / "data" / "processed" / "string_v12_0"
META = ROOT / "data" / "metadata"

PROCESSED.mkdir(parents=True, exist_ok=True)
META.mkdir(parents=True, exist_ok=True)

FILES = {
    "functional": RAW / "4932.protein.links.full.v12.0.txt.gz",
    "physical": RAW / "4932.protein.physical.links.full.v12.0.txt.gz",
    "info": RAW / "4932.protein.info.v12.0.txt.gz",
    "aliases": RAW / "4932.protein.aliases.v12.0.txt.gz",
}

missing = [str(path) for path in FILES.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Please download the following files from STRING and locate to data/raw/string_v12_0/:\n"
        + "\n".join(missing)
    )

def sha256(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

file_report = pd.DataFrame([
    {
        "role": role,
        "filename": path.name,
        "size_MB": round(path.stat().st_size / 1_000_000, 2),
        "sha256": sha256(path),
    }
    for role, path in FILES.items()
])

display(file_report)

,role,filename,size_MB,sha256
0,functional,4932.protein.links.full.v12.0.txt.gz,30.94,1362829d6685ad1c03e913204862ec369b86075eaba142...
1,physical,4932.protein.physical.links.full.v12.0.txt.gz,2.57,0db5126a1fe8eb3e56cb574e946e0f66cc43d68c3fca62...
2,info,4932.protein.info.v12.0.txt.gz,0.55,ad69b4594f35cf6ffeb52afbfc3fe30d639bb76e0b883b...
3,aliases,4932.protein.aliases.v12.0.txt.gz,1.61,5991f48fb4cf5f1d34948010fc2a074962a0fd1751e7f9...


In [2]:
def clean_columns(frame):
    frame = frame.copy()
    frame.columns = [str(column).lstrip("#") for column in frame.columns]
    return frame

# links file separates by empty string; info and aliases file by tab
for layer in ("functional", "physical"):
    header = pd.read_csv(
        FILES[layer],
        sep=r"\s+",
        compression="gzip",
        nrows=0,
    )
    print(layer, clean_columns(header).columns.tolist())

for role in ("info", "aliases"):
    header = pd.read_csv(
        FILES[role],
        sep=r"\t",
        compression="gzip",
        nrows=0,
    )
    print(role, clean_columns(header).columns.tolist())

functional ['protein1', 'protein2', 'neighborhood', 'neighborhood_transferred', 'fusion', 'cooccurence', 'homology', 'coexpression', 'coexpression_transferred', 'experiments', 'experiments_transferred', 'database', 'database_transferred', 'textmining', 'textmining_transferred', 'combined_score']
physical ['protein1', 'protein2', 'homology', 'experiments', 'experiments_transferred', 'database', 'database_transferred', 'textmining', 'textmining_transferred', 'combined_score']
info ['string_protein_id', 'preferred_name', 'protein_size', 'annotation']
aliases ['string_protein_id', 'alias', 'source']


/var/folders/kc/4swzx7w979z6w9js5c61gt7h0000gn/T/ipykernel_85627/1561477670.py:17: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  header = pd.read_csv(
/var/folders/kc/4swzx7w979z6w9js5c61gt7h0000gn/T/ipykernel_85627/1561477670.py:17: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  header = pd.read_csv(


In [4]:
# Read protein info and aliases
info = clean_columns(pd.read_csv(FILES["info"], sep="\t", compression="gzip"))
aliases = clean_columns(pd.read_csv(FILES["aliases"], sep="\t", compression="gzip"))

required_info = {"string_protein_id", "preferred_name"}
required_aliases = {"string_protein_id", "alias"}

assert required_info <= set(info.columns), info.columns.tolist()
assert required_aliases <= set(aliases.columns), aliases.columns.tolist()
assert info["string_protein_id"].is_unique

info["string_protein_id"] = info["string_protein_id"].astype(str)
aliases["string_protein_id"] = aliases["string_protein_id"].astype(str)
aliases["alias"] = aliases["alias"].astype(str)

print("STRING proteins:", len(info))
print("Alias rows:", len(aliases))
display(info.head())
display(aliases.head())

STRING proteins: 6600
Alias rows: 287378


,string_protein_id,preferred_name,protein_size,annotation
0,4932.Q0010,Q0010,128,"Putative uncharacterized protein Q0010, mitoch..."
1,4932.Q0017,Q0017,53,"Putative uncharacterized protein Q0017, mitoch..."
2,4932.Q0032,Q0032,96,"Putative uncharacterized protein Q0032, mitoch..."
3,4932.Q0045,COX1,534,Subunit I of cytochrome c oxidase (Complex IV)...
4,4932.Q0050,AI1,834,Putative COX1/OXI3 intron 1 protein; Reverse t...


,string_protein_id,alias,source
0,4932.Q0010,4932.Q0010,Ensembl_STRING
1,4932.Q0010,KP263414,Ensembl_EMBL
2,4932.Q0010,L000004921,SGD_ID
3,4932.Q0010,ORF6,SGD_SYNONYM
4,4932.Q0010,ORF6,UniProt_GN_ORFNames


In [7]:
# setup STRING ID -> ORF mapping
# Common yeast core gene ID, e.g. YMR161W, YNL064C
ORF_PATTERN = re.compile(r"^Y[A-P][LR]\d{3}[CW](?:-[A-Z])?$")

alias_orfs = aliases.loc[
    aliases["alias"].map(lambda value: bool(ORF_PATTERN.fullmatch(value))),
    ["string_protein_id", "alias"],
]

alias_dict = (
    alias_orfs
    .groupby("string_protein_id")["alias"]
    .agg(lambda x: sorted(set(x)))
    .to_dict()
)

id_map = info[["string_protein_id", "preferred_name"]].copy()
id_map["id_suffix"] = id_map["string_protein_id"].str.split(".", n=1).str[-1]
id_map["orf_aliases"] = id_map["string_protein_id"].map(
    lambda string_id: alias_dict.get(string_id, [])
)

def decide_orf(row):
    direct = row["id_suffix"] if ORF_PATTERN.fullmatch(row["id_suffix"]) else None
    candidates = row["orf_aliases"]

    if direct is not None:
        if candidates and direct not in candidates:
            return pd.Series([None, "conflict"])
        return pd.Series([direct, "string_id"])

    if len(candidates) == 1:
        return pd.Series([candidates[0], "unique_alias"])
    if len(candidates) > 1:
        return pd.Series([None, "ambiguous_alias"])
    return pd.Series([None, "unmapped"])

id_map[["orf", "mapping_status"]] = id_map.apply(decide_orf, axis=1)

display(id_map["mapping_status"].value_counts())
display(id_map.loc[id_map["mapping_status"].isin(
    ["conflict", "ambiguous_alias"]
)].head(20))

# If STRING ID many-one ORF, artificial match
duplicate_orfs = id_map.loc[id_map["orf"].notna()].groupby("orf").size()
display(duplicate_orfs.loc[duplicate_orfs > 1].head(20))

hlj1_hits = id_map.loc[id_map["orf"].eq("YMR161W")]
assert len(hlj1_hits) == 1, (
    "HLJ1/YMR161W has no distinct map; check aliases in id_map first"
)
HLJ1_ID = hlj1_hits.iloc[0]["string_protein_id"]

print("HLJ1 STRING ID:", HLJ1_ID)
display(id_map.loc[
    id_map["preferred_name"].str.upper().isin(["HLJ1", "KAR2", "YDJ1"])
    | id_map["orf"].eq("YMR161W")
])

id_map.to_csv(PROCESSED / "string_id_to_orf.csv", index=False)


mapping_status
string_id    6572
unmapped       28
Name: count, dtype: int64

,string_protein_id,preferred_name,id_suffix,orf_aliases,orf,mapping_status


Series([], dtype: int64)

HLJ1 STRING ID: 4932.YMR161W


,string_protein_id,preferred_name,id_suffix,orf_aliases,orf,mapping_status
3260,4932.YJL034W,KAR2,YJL034W,[YJL034W],YJL034W,string_id
4871,4932.YMR161W,HLJ1,YMR161W,[YMR161W],YMR161W,string_id
5121,4932.YNL064C,YDJ1,YNL064C,[YNL064C],YNL064C,string_id


The undirected interaction may record both $(i,j)$ and $(j,i)$. In the graph they are supposed to be a single edge: $\{i,j\}=\{j,i\}$.

For each network layer, preserve the edges such that `combined_score >= 400` to support three categories of analysis: $\tau=0.4,0.7,0.9$. Remove self-loops, and combine the reverse repeated records. If the same undirected pair of proteins has different score, stop and check original data.

In [9]:
def prepare_edges(path, min_score=400, chunk_size=150_000):
    kept_chunks = []
    raw_rows, self_loops = 0, 0

    reader = pd.read_csv(path, sep=r"\s+", compression="gzip", chunksize=chunk_size)

    for chunk in reader:
        chunk = clean_columns(chunk)
        required = {"protein1", "protein2", "combined_score"}
        if not required <= set(chunk.columns):
            raise ValueError(
                f"Missing necessary value in {path.name}; actual value: {chunk.columns.tolist()}"
            )

        raw_rows += len(chunk)
        chunk["combined_score"] = pd.to_numeric(
            chunk["combined_score"], errors="raise"
        )

        if not chunk["protein1"].astype(str).str.startswith("4932.").all():
            raise ValueError("protein1 of non-4932 specie found")
        if not chunk["protein2"].astype(str).str.startswith("4932.").all():
            raise ValueError("protein2 of non-4932 specie found")

        chunk = chunk.loc[chunk["combined_score"] >= min_score].copy()
        self_loops += int((chunk["protein1"] == chunk["protein2"]).sum())
        chunk = chunk.loc[chunk["protein1"] != chunk["protein2"]].copy()

        left, right = chunk["protein1"].astype(str), chunk["protein2"].astype(str)
        left_first = left < right

        chunk["u"] = left.where(left_first, right)
        chunk["v"] = right.where(left_first, left)
        kept_chunks.append(chunk)

    if not kept_chunks:
        raise ValueError(f"{path.name} did not reach the minimum score")

    edges = pd.concat(kept_chunks, ignore_index=True)

    disagreement = (
        edges.groupby(["u", "v"])["combined_score"]
        .nunique()
        .gt(1)
        .sum()
    )
    if disagreement:
        raise ValueError(
            f"{path.name}: {disagreement} edge with repeated pair has different combined_score"
        )

    rows_before_dedup = len(edges)
    edges = (
        edges.sort_values(["u", "v", "combined_score"])
        .drop_duplicates(["u", "v"], keep="last")
        .reset_index(drop=True)
    )

    if not edges["combined_score"].between(400, 1000).all():
        raise ValueError("combined_score exceeded expectation")

    report = {
        "raw_rows": raw_rows,
        "self_loops_after_score_filter": self_loops,
        "rows_score_ge_400_before_dedup": rows_before_dedup,
        "unique_undirected_edges_ge_400": len(edges),
        "reverse_or_duplicate_rows_removed": rows_before_dedup - len(edges),
    }
    return edges, report

functional_edges, functional_report = prepare_edges(FILES["functional"])
physical_edges, physical_report = prepare_edges(FILES["physical"])

edge_tables = {
    "functional": functional_edges,
    "physical": physical_edges,
}

display(pd.DataFrame({
    "functional": functional_report,
    "physical": physical_report
}).T)

known_ids = set(info["string_protein_id"])
for layer, edges in edge_tables.items():
    endpoints = set(edges["u"]) | set(edges["v"])
    unknown = endpoints - known_ids
    assert not unknown, f"{layer} has {len(unknown)} ID not in info"

    output = PROCESSED / f"{layer}_edges_score_ge_400.csv.gz"
    edges.to_csv(output, index=False, compression="gzip")
    print(layer, "->", output, "| columns:", edges.columns.tolist())

,raw_rows,self_loops_after_score_filter,rows_score_ge_400_before_dedup,unique_undirected_edges_ge_400,reverse_or_duplicate_rows_removed
functional,2824842,0,562356,281178,281178
physical,352606,0,140402,70201,70201


functional -> /Users/kennyyu/math3888-stream-2-group-k/data/processed/string_v12_0/functional_edges_score_ge_400.csv.gz | columns: ['protein1', 'protein2', 'neighborhood', 'neighborhood_transferred', 'fusion', 'cooccurence', 'homology', 'coexpression', 'coexpression_transferred', 'experiments', 'experiments_transferred', 'database', 'database_transferred', 'textmining', 'textmining_transferred', 'combined_score', 'u', 'v']
physical -> /Users/kennyyu/math3888-stream-2-group-k/data/processed/string_v12_0/physical_edges_score_ge_400.csv.gz | columns: ['protein1', 'protein2', 'homology', 'experiments', 'experiments_transferred', 'database', 'database_transferred', 'textmining', 'textmining_transferred', 'combined_score', 'u', 'v']


In [11]:
# Build networks by thresholds, check the coverage of HLJ1
def build_graph(edges, info, cutoff):
    graph = nx.Graph()
    graph.add_nodes_from(info["string_protein_id"])

    selected = edges.loc[
        edges["combined_score"] >= cutoff,
        ["u", "v", "combined_score"],
    ]

    graph.add_weighted_edges_from(
        (u, v, score / 1000)
        for u, v, score in selected.itertuples(index=False, name=None)
    )
    return graph

summary_rows = []
primary_graphs = {}
for layer, edges in edge_tables.items():
    for cutoff in (400, 700, 900):
        graph = build_graph(edges, info, cutoff)

        summary_rows.append({
            "layer": layer,
            "threshold": cutoff / 1000,
            "nodes": graph.number_of_nodes(),
            "edges": graph.number_of_edges(),
            "isolates": nx.number_of_isolates(graph),
            "components": nx.number_connected_components(graph),
            "HLJ1_degree": graph.degree(HLJ1_ID),
            "HLJ1_component_size": len(nx.node_connected_component(graph, HLJ1_ID))
        })

        if cutoff == 700:
            primary_graphs[layer] = graph

network_summary = pd.DataFrame(summary_rows)
display(network_summary)
network_summary.to_csv(PROCESSED / "network_threshold_summary.csv", index=False)

,layer,threshold,nodes,edges,isolates,components,HLJ1_degree,HLJ1_component_size
0,functional,0.4,6600,281178,308,310,73,6290
1,functional,0.7,6600,104188,809,840,25,5716
2,functional,0.9,6600,53176,1888,2022,4,4312
3,physical,0.4,6600,70201,1775,1801,0,1
4,physical,0.7,6600,43030,3216,3432,0,1
5,physical,0.9,6600,21941,3798,4074,0,1


In [12]:
# Construct ERAD seed set template
seed_path = META / "erad_seeds.csv"

if not seed_path.exists():
    template = pd.DataFrame([{
        "orf": "YMR161W",
        "set": "core",
        "source_url": "https://www.kegg.jp/entry/sce:YMR161W",
        "rationale": "HLJ1; starting anchor for ERAD-related analysis",
    }])
    template.to_csv(seed_path, index=False)
    print("Template generated: ", seed_path)

print(
    f"Next step: manually add other ERAD proteins according to sources e.g. KEGG/SGD;"
    f"write on each row: ORF, core/broad category, link to the source and reason(s) that it was added"
)
display(pd.read_csv(seed_path))

Template generated:  /Users/kennyyu/math3888-stream-2-group-k/data/metadata/erad_seeds.csv
Next step: manually add other ERAD proteins according to sources e.g. KEGG/SGD;write on each row: ORF, core/broad category, link to the source and reason(s) that it was added


,orf,set,source_url,rationale
0,YMR161W,core,https://www.kegg.jp/entry/sce:YMR161W,HLJ1; starting anchor for ERAD-related analysis


In [17]:
# Add a source-backed starting set for manual review
pathway_url = "https://www.kegg.jp/entry/sce04141"

proposed_seeds = pd.DataFrame([
    {
        "orf": "YOL013C",
        "set": "core",
        "source_url": pathway_url,
        "rationale": "HRD1: ER-associated E3 ubiquitin ligase",
    },
    {
        "orf": "YLR207W",
        "set": "core",
        "source_url": pathway_url,
        "rationale": "HRD3: HRD ubiquitin-ligase complex subunit",
    },
    {
        "orf": "YBR201W",
        "set": "core",
        "source_url": pathway_url,
        "rationale": "DER1: derlin associated with ER protein degradation",
    },
    {
        "orf": "YMR022W",
        "set": "core",
        "source_url": pathway_url,
        "rationale": "UBC7: E2 ubiquitin-conjugating protein",
    },
    {
        "orf": "YMR264W",
        "set": "core",
        "source_url": pathway_url,
        "rationale": "CUE1: ERAD-associated component listed in sce04141",
    },
])

existing_seeds = pd.read_csv(seed_path, dtype=str).fillna("")
new_rows = proposed_seeds.loc[
    ~proposed_seeds["orf"].isin(existing_seeds["orf"])
]

updated_seeds = pd.concat(
    [existing_seeds, new_rows],
    ignore_index=True,
)
updated_seeds.to_csv(seed_path, index=False)

print(f"Added {len(new_rows)} rows; total seeds: {len(updated_seeds)}")
display(updated_seeds)

# Preview whether each proposed ORF has a STRING mapping.
display(
    id_map.loc[
        id_map["orf"].isin(updated_seeds["orf"]),
        ["orf", "preferred_name", "string_protein_id", "mapping_status"],
    ].sort_values("orf")
)

Added 0 rows; total seeds: 6


,orf,set,source_url,rationale
0,YMR161W,core,https://www.kegg.jp/entry/sce:YMR161W,HLJ1; starting anchor for ERAD-related analysis
1,YOL013C,core,https://www.kegg.jp/entry/sce04141,HRD1: ER-associated E3 ubiquitin ligase
2,YLR207W,core,https://www.kegg.jp/entry/sce04141,HRD3: HRD ubiquitin-ligase complex subunit
3,YBR201W,core,https://www.kegg.jp/entry/sce04141,DER1: derlin associated with ER protein degrad...
4,YMR022W,core,https://www.kegg.jp/entry/sce04141,UBC7: E2 ubiquitin-conjugating protein
5,YMR264W,core,https://www.kegg.jp/entry/sce04141,CUE1: ERAD-associated component listed in sce0...


,orf,preferred_name,string_protein_id,mapping_status
492,YBR201W,DER1,4932.YBR201W,string_id
4269,YLR207W,HRD3,4932.YLR207W,string_id
4722,YMR022W,UBC7,4932.YMR022W,string_id
4871,YMR161W,HLJ1,4932.YMR161W,string_id
4984,YMR264W,CUE1,4932.YMR264W,string_id
5504,YOL013C,HRD1,4932.YOL013C,string_id


In [18]:
# Check seed mappings and freeze the input record

seeds = pd.read_csv(seed_path, dtype=str).fillna("")

required_seed_columns = {"orf", "set", "source_url", "rationale"}
assert required_seed_columns <= set(seeds.columns)
assert seeds["orf"].is_unique, "Duplicate ORF exists in seed table"

core_seeds = seeds.loc[seeds["set"] == "core"].copy()
assert "YMR161W" in set(core_seeds["orf"])

# Only consider mappings for the ORFs we actually selected as core seeds.
candidates = (
    id_map.loc[
        id_map["orf"].isin(core_seeds["orf"]),
        ["orf", "string_protein_id", "mapping_status"],
    ]
    .dropna(subset=["orf", "string_protein_id"])
    .drop_duplicates(subset=["orf", "string_protein_id"])
)

counts = candidates.groupby("orf")["string_protein_id"].nunique()
ambiguous_orfs = set(counts.loc[counts > 1].index)

if ambiguous_orfs:
    print("These core seeds have multiple STRING IDs; inspect them manually:")
    display(
        candidates.loc[candidates["orf"].isin(ambiguous_orfs)]
        .sort_values(["orf", "string_protein_id"])
    )

# Only unambiguous mappings may enter the one-to-one merge.
unique_candidates = candidates.loc[
    ~candidates["orf"].isin(ambiguous_orfs)
].copy()

mapped_seeds = core_seeds.merge(
    unique_candidates,
    on="orf",
    how="left",
    validate="one_to_one",
)

mapped_seeds["mapping_issue"] = "ok"
mapped_seeds.loc[
    mapped_seeds["orf"].isin(ambiguous_orfs), "mapping_issue"
] = "ambiguous"
mapped_seeds.loc[
    mapped_seeds["string_protein_id"].isna()
    & ~mapped_seeds["orf"].isin(ambiguous_orfs),
    "mapping_issue",
] = "unmapped"

display(mapped_seeds)

issues = mapped_seeds.loc[mapped_seeds["mapping_issue"] != "ok"]
missing_sources = core_seeds.loc[
    core_seeds["source_url"].eq("") | core_seeds["rationale"].eq("")
]

if len(core_seeds) < 3:
    print("Core seed set is incomplete: at least 3 proteins are needed.")
    print("After removing HLJ1, pathway efficiency requires |S| >= 2.")

if not issues.empty:
    print("Resolve these seed mappings before freezing the design:")
    display(issues)

if not missing_sources.empty:
    print("Add a source URL and rationale for these seeds:")
    display(missing_sources)

ready_to_freeze = (
    len(core_seeds) >= 3
    and issues.empty
    and missing_sources.empty
)

if ready_to_freeze:
    hlj1_mapped_id = mapped_seeds.loc[
        mapped_seeds["orf"] == "YMR161W", "string_protein_id"
    ].iloc[0]
    assert hlj1_mapped_id == HLJ1_ID

    design = {
        "string_release": "12.0",
        "taxon_id": 4932,
        "primary_layer": "functional",
        "sensitivity_layer": "physical",
        "primary_score_threshold": 700,
        "sensitivity_thresholds": [400, 900],
        "hlj1_string_id": HLJ1_ID,
        "core_seed_orfs": sorted(core_seeds["orf"].tolist()),
        "raw_file_sha256": {
            role: sha256(path) for role, path in FILES.items()
        },
        "seed_table_sha256": sha256(seed_path),
    }

    design_path = META / "string_analysis_design_v12_0.json"

    if design_path.exists():
        old_design = json.loads(design_path.read_text(encoding="utf-8"))
        assert old_design == design, (
            "Frozen inputs or seed table changed. Record the reason "
            "and save a revised design version."
        )
        print("Current inputs match the frozen design.")
    else:
        design_path.write_text(
            json.dumps(design, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )
        print("Frozen design saved:", design_path)

,orf,set,source_url,rationale,string_protein_id,mapping_status,mapping_issue
0,YMR161W,core,https://www.kegg.jp/entry/sce:YMR161W,HLJ1; starting anchor for ERAD-related analysis,4932.YMR161W,string_id,ok
1,YOL013C,core,https://www.kegg.jp/entry/sce04141,HRD1: ER-associated E3 ubiquitin ligase,4932.YOL013C,string_id,ok
2,YLR207W,core,https://www.kegg.jp/entry/sce04141,HRD3: HRD ubiquitin-ligase complex subunit,4932.YLR207W,string_id,ok
3,YBR201W,core,https://www.kegg.jp/entry/sce04141,DER1: derlin associated with ER protein degrad...,4932.YBR201W,string_id,ok
4,YMR022W,core,https://www.kegg.jp/entry/sce04141,UBC7: E2 ubiquitin-conjugating protein,4932.YMR022W,string_id,ok
5,YMR264W,core,https://www.kegg.jp/entry/sce04141,CUE1: ERAD-associated component listed in sce0...,4932.YMR264W,string_id,ok


Frozen design saved: /Users/kennyyu/math3888-stream-2-group-k/data/metadata/string_analysis_design_v12_0.json
